In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/arXiv_scientific_dataset.csv")

# keep required columns
df = df[[
    "title",
    "summary",
    "category_code",
    "authors",
    "published_date"
]]

# rename (for consistency)
df.rename(columns={
    "title": "titles",
    "summary": "summaries",
    "category_code": "terms"
}, inplace=True)

# drop nulls
df = df.dropna()

print("Initial size:", len(df))

Initial size: 136238


In [14]:
# convert labels → list
df["terms"] = df["terms"].apply(lambda x: x.split())

# remove rare labels (IMPORTANT)
from collections import Counter

all_labels = df["terms"].explode()
label_counts = Counter(all_labels)

min_count = 15   # 🔥 best balance for large dataset
valid_labels = {l for l, c in label_counts.items() if c >= min_count}

df["terms"] = df["terms"].apply(lambda x: [l for l in x if l in valid_labels])
df = df[df["terms"].map(len) > 0]

print("After label filtering:", len(df))

df.to_csv("../data/processed_dataset.csv", index=False)

After label filtering: 135939


In [3]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
val_df = test_df.sample(frac=0.5, random_state=42)
test_df = test_df.drop(val_df.index)

print(len(train_df), len(val_df), len(test_df))

122345 6797 6797


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.85,
    stop_words="english",
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df["summaries"])
X_val = vectorizer.transform(val_df["summaries"])
X_test = vectorizer.transform(test_df["summaries"])

In [6]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(train_df["terms"])
y_val = mlb.transform(val_df["terms"])
y_test = mlb.transform(test_df["terms"])

print("Total labels:", len(mlb.classes_))

Total labels: 80


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier

# Logistic Regression
model_lr = OneVsRestClassifier(
    LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1)
)
model_lr.fit(X_train, y_train)

# Naive Bayes
model_nb = OneVsRestClassifier(MultinomialNB())
model_nb.fit(X_train, y_train)

,estimator,MultinomialNB()
,n_jobs,None
,verbose,0
,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [8]:
probs_lr = model_lr.predict_proba(X_test)
probs_nb = model_nb.predict_proba(X_test)

# weighted ensemble
final_probs = (0.6 * probs_lr) + (0.4 * probs_nb)

# light sharpening (important)
final_probs = np.power(final_probs, 1.2)

In [9]:
# def predict_top_k_matrix(probs, k=1):
#     y_pred = np.zeros_like(probs)
    
#     for i in range(probs.shape[0]):
#         top_idx = np.argsort(probs[i])[-k:]
#         y_pred[i, top_idx] = 1
        
#     return y_pred

# k = 1
# y_pred = predict_top_k_matrix(final_probs, k=k)

def predict_dynamic_k(probs, threshold=0.3, max_k=2):
    y_pred = np.zeros_like(probs)
    
    for i in range(probs.shape[0]):
        idx = np.where(probs[i] >= threshold)[0]
        
        if len(idx) == 0:
            idx = np.argsort(probs[i])[-1:]
        elif len(idx) > max_k:
            idx = np.argsort(probs[i])[-max_k:]
            
        y_pred[i, idx] = 1
        
    return y_pred

y_pred = predict_dynamic_k(final_probs, threshold=0.3, max_k=2)

In [10]:
from sklearn.metrics import f1_score, precision_score, recall_score

print("F1 micro:", f1_score(y_test, y_pred, average="micro"))
print("F1 macro:", f1_score(y_test, y_pred, average="macro"))
print("Precision:", precision_score(y_test, y_pred, average="micro"))
print("Recall:", recall_score(y_test, y_pred, average="micro"))

print("Avg labels per sample:", y_pred.sum(axis=1).mean())

F1 micro: 0.7103213242453749
F1 macro: 0.25143073286178097
Precision: 0.60570835495589
Recall: 0.8586140944534353
Avg labels per sample: 1.417537148742092


C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [12]:
import pickle
import os

os.makedirs("../models", exist_ok=True)

pickle.dump(model_lr, open("../models/model_lr.pkl", "wb"))
pickle.dump(model_nb, open("../models/model_nb.pkl", "wb"))
pickle.dump(vectorizer, open("../models/vectorizer.pkl", "wb"))
pickle.dump(mlb, open("../models/mlb.pkl", "wb"))

print("Models saved!")

Models saved!
